In [ ]:
!pip install lightgbm imbalanced-learn pytorch-tabnet catboost scikeras -q

In [ ]:
import time, os, io, zipfile, numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt
from scipy.interpolate import interp1d
from sklearn.model_selection import GroupKFold, RandomizedSearchCV, train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score, log_loss, precision_recall_curve, precision_score, recall_score,
                             average_precision_score, f1_score, roc_curve, auc, RocCurveDisplay)
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.base import clone as sklearn_clone
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, MinMaxScaler, label_binarize
from sklearn.feature_selection import mutual_info_classif
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, ClassifierMixin
import seaborn as sns
import shap
import joblib
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Input, Dense, Dropout, BatchNormalization,
                                     LSTM, Conv1D, MaxPooling1D)
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import to_categorical
from pytorch_tabnet.tab_model import TabNetClassifier
from pytorch_tabnet.callbacks import Callback
import zipfile
import tempfile
import shutil
import random
import warnings

warnings.filterwarnings('ignore')

In [ ]:
PARK_ZIP = "/content/Clean Dataset – Parkinson’s Group.zip"
CTRL_ZIP = "/content/Clean Dataset – Control Group.zip"
VOL_ZIP = "/content/Clean Dataset – Voluntary Group.zip"
FS = 66.67
WINDOW_SIZE = int(round(FS * 1.0))   # 1 second
LOWCUT = 0.5
HIGHCUT = 20.0
RANDOM_STATE = 42
SIGNAL_COLS = ["AX", "AY", "AZ", "GX", "GY", "GZ"]
all_models_metrics = {}

In [ ]:
def bandpass(x, fs=FS, low=LOWCUT, high=HIGHCUT, order=4):
    nyq = fs / 2
    b, a = butter(order, [low/nyq, high/nyq], btype="band")
    return filtfilt(b, a, x)

def resample_df(df, fs=FS):
    t = (df["T"].values - df["T"].values[0]) / 1000.0
    new_t = np.arange(0, t[-1], 1/fs)
    out = pd.DataFrame({"T": new_t * 1000})
    for c in SIGNAL_COLS:
        f = interp1d(t, df[c].values, kind="linear", fill_value="extrapolate")
        out[c] = f(new_t)
    return out

def fft_top2(x, fs=FS):
    x = x - np.mean(x)
    freqs = np.fft.rfftfreq(len(x), d=1/fs)
    mag = np.abs(np.fft.rfft(x))
    mask = (freqs >= LOWCUT) & (freqs <= HIGHCUT)
    freqs, mag = freqs[mask], mag[mask]
    idx = np.argsort(mag)[::-1]
    i1 = idx[0]
    i2 = idx[1] if len(idx) > 1 else idx[0]
    return freqs[i1], mag[i1], freqs[i2], mag[i2]

def extract_features(win):
    feats = {}
    for c in SIGNAL_COLS:
        x = win[c].values
        f1, a1, f2, a2 = fft_top2(x)
        feats[f"{c}_mean"] = np.mean(x)
        feats[f"{c}_std"] = np.std(x)
        feats[f"{c}_median"] = np.median(x)
        feats[f"{c}_q1"] = np.percentile(x, 25)
        feats[f"{c}_q3"] = np.percentile(x, 75)
        feats[f"{c}_min"] = np.min(x)
        feats[f"{c}_max"] = np.max(x)
        feats[f"{c}_peak1_freq"] = f1
        feats[f"{c}_peak1_amp"] = a1
        feats[f"{c}_peak2_freq"] = f2
        feats[f"{c}_peak2_amp"] = a2
    return feats

In [ ]:
def load_zip_features(zip_path, default_label):
    rows = []
    with zipfile.ZipFile(zip_path, "r") as z:
        files = [f for f in z.namelist() if f.lower().endswith(".csv")]
        for file in files:
            df = pd.read_csv(io.BytesIO(z.read(file)))
            df.columns = ["T","AX","AY","AZ","GX","GY","GZ"]
            df = resample_df(df)
            for c in SIGNAL_COLS:
                df[c] = bandpass(df[c].values)
            file_id = os.path.splitext(os.path.basename(file))[0]
            final_label = default_label
            for start in range(0, len(df) - WINDOW_SIZE + 1, WINDOW_SIZE):
                win = df.iloc[start:start + WINDOW_SIZE]
                feats = extract_features(win)
                feats["file_id"] = file_id
                feats["label"] = final_label
                rows.append(feats)
    return pd.DataFrame(rows)

# Load data
park_features = load_zip_features(PARK_ZIP, 1)
ctrl_features = load_zip_features(CTRL_ZIP, 0)
vol_features = load_zip_features(VOL_ZIP, 2)

feature_df = pd.concat([park_features, ctrl_features, vol_features], ignore_index=True)
feature_df = feature_df.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

# Separate features and labels
X = feature_df.drop(columns=["file_id", "label"])
y = feature_df["label"].values
groups = feature_df["file_id"].values
cv = GroupKFold(n_splits=4)

print("Dataset shape:", X.shape)
print("Class distribution:", dict(zip(*np.unique(y, return_counts=True))))
print("Number of unique files:", feature_df["file_id"].nunique())

In [ ]:
def evaluate_model(model, X, y, groups, title="Model", training_time=None, history=None):
    y_true_all = np.full(len(y), -1)
    y_pred_all = np.full(len(y), -1)
    y_proba_all = None
    model_copy = None

    for train_idx, test_idx in cv.split(X, y, groups):
        X_train = X.iloc[train_idx] if hasattr(X, "iloc") else X[train_idx]
        X_test = X.iloc[test_idx] if hasattr(X, "iloc") else X[test_idx]
        y_train = y[train_idx]
        y_test = y[test_idx]
        model_copy = clone_model(model)
        model_copy.fit(X_train, y_train)
        y_pred = model_copy.predict(X_test)
        if y_pred.ndim > 1:
            y_pred = y_pred.ravel()
        y_pred_all[test_idx] = y_pred
        y_true_all[test_idx] = y_test
        if hasattr(model_copy, "predict_proba"):
            proba = model_copy.predict_proba(X_test)
            if y_proba_all is None:
                y_proba_all = np.zeros((len(y), proba.shape[1]))
            y_proba_all[test_idx] = proba

    valid = (y_true_all != -1)
    y_true_all = y[valid]
    y_pred_all = y_pred_all[valid]
    y_proba_all = y_proba_all[valid] if y_proba_all is not None else None

    # ----- WINDOW LEVEL -----
    print(f"\n===== {title} | WINDOW LEVEL =====")
    print(classification_report(y_true_all, y_pred_all,
                                target_names=["Non-Tremor","Tremor","Voluntary"]))
    cm = confusion_matrix(y_true_all, y_pred_all)
    cm_percent = cm.astype('float') / cm.sum(axis=1, keepdims=True) * 100
    plt.figure(figsize=(6,5))
    sns.heatmap(cm_percent, annot=True, fmt='.1f', cmap='Blues',
                xticklabels=["Non-Tremor","Tremor","Voluntary"],
                yticklabels=["Non-Tremor","Tremor","Voluntary"])
    plt.title(f"{title} - Window Level Confusion Matrix (%)")
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()

    if y_proba_all is not None:
        plot_multiclass_roc(y_true_all, y_proba_all, ["Non-Tremor","Tremor","Voluntary"], level="Window")
        plot_precision_recall_curves(y_true_all, y_proba_all, ["Non-Tremor","Tremor","Voluntary"], level="Window")
        acc, loss, precision, recall, f1, fnr = calculate_metrics(y_true_all, y_pred_all, y_proba_all)
        print(f"\n===== {title} | WINDOW LEVEL METRICS =====")
        print(f"Accuracy: {acc:.4f} | Log Loss: {loss:.4f} | Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f} | FNR: {fnr:.4f}")

    # ----- FILE LEVEL (Majority voting and average probabilities) -----
    vote_df = pd.DataFrame({"file_id": groups[valid], "y_true": y[valid], "y_pred": y_pred_all})
    if y_proba_all is not None:
        vote_df['proba'] = list(y_proba_all)
    file_pred = vote_df.groupby("file_id")["y_pred"].agg(lambda x: x.mode().iloc[0])
    file_true = vote_df.groupby("file_id")["y_true"].agg(lambda x: x.mode().iloc[0])
    if y_proba_all is not None:
        file_proba = vote_df.groupby("file_id")["proba"].apply(lambda x: np.mean(np.vstack(x), axis=0)).values
        file_proba = np.vstack(file_proba)
    else:
        file_proba = None

    print(f"\n===== {title} | FILE LEVEL =====")
    print(classification_report(file_true, file_pred,
                                target_names=["Non-Tremor","Tremor","Voluntary"]))
    cm_file = confusion_matrix(file_true, file_pred)
    cm_file_percent = cm_file.astype('float') / cm_file.sum(axis=1, keepdims=True) * 100
    plt.figure(figsize=(6,5))
    sns.heatmap(cm_file_percent, annot=True, fmt='.1f', cmap='Blues',
                xticklabels=["Non-Tremor","Tremor","Voluntary"],
                yticklabels=["Non-Tremor","Tremor","Voluntary"])
    plt.title(f"{title} - File Level Confusion Matrix (%)")
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.show()

    if file_proba is not None:
        plot_multiclass_roc(file_true, file_proba, ["Non-Tremor","Tremor","Voluntary"], level="File")
        plot_precision_recall_curves(file_true, file_proba, ["Non-Tremor","Tremor","Voluntary"], level="File")
        acc_f, loss_f, precision_f, recall_f, f1_f, fnr_f = calculate_metrics(file_true, file_pred, file_proba)
        print(f"\n===== {title} | FILE LEVEL METRICS =====")
        print(f"Accuracy: {acc_f:.4f} | Log Loss: {loss_f:.4f} | Precision: {precision_f:.4f} | Recall: {recall_f:.4f} | F1: {f1_f:.4f} | FNR: {fnr_f:.4f}")
    else:
        acc_f = accuracy_score(file_true, file_pred)
        precision_f = precision_score(file_true, file_pred, average='macro', zero_division=0)
        recall_f = recall_score(file_true, file_pred, average='macro', zero_division=0)
        f1_f = f1_score(file_true, file_pred, average='macro', zero_division=0)
        fnr_f = 1 - recall_f
        loss_f = np.nan
        print(f"\n===== {title} | FILE LEVEL METRICS (no probabilities) =====")
        print(f"Accuracy: {acc_f:.4f} | Precision: {precision_f:.4f} | Recall: {recall_f:.4f} | F1: {f1_f:.4f} | FNR: {fnr_f:.4f}")

    # Store metrics in global dictionary (both levels)
    if 'all_models_metrics' not in globals():
        global all_models_metrics
        all_models_metrics = {}
    all_models_metrics[f"{title}_window"] = {
        'Accuracy': acc, 'Loss': loss, 'Precision': precision,
        'Recall': recall, 'F1': f1, 'FNR': fnr,
        'Training Time': training_time
    }
    all_models_metrics[f"{title}_file"] = {
        'Accuracy': acc_f, 'Loss': loss_f, 'Precision': precision_f,
        'Recall': recall_f, 'F1': f1_f, 'FNR': fnr_f,
        'Training Time': training_time
    }

    if history is not None:
        plot_training_history(history, title, level="Window")

    X_file, y_file, _ = get_file_level_features(X, y, groups)
    if "MLP" in title or "LSTM" in title or "TabNet" in title or "CNN+LSTM" in title:
        X_train_f, X_val_f, y_train_f, y_val_f = train_test_split(X_file, y_file, test_size=0.2, random_state=RANDOM_STATE)

        if "MLP" in title:
            def build_keras_mlp(input_dim):
                model = Sequential([
                    Input(shape=(input_dim,)),
                    Dense(256, activation='relu'), BatchNormalization(), Dropout(0.3),
                    Dense(128, activation='relu'), BatchNormalization(), Dropout(0.3),
                    Dense(64, activation='relu'), Dropout(0.2),
                    Dense(3, activation='softmax')
                ])
                model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
                return model
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train_f)
            X_val_scaled = scaler.transform(X_val_f)
            file_model = build_keras_mlp(X_train_scaled.shape[1])
            hist_file = file_model.fit(X_train_scaled, y_train_f, validation_data=(X_val_scaled, y_val_f),
                                       epochs=100, batch_size=64, verbose=0,
                                       callbacks=[EarlyStopping(patience=10, restore_best_weights=True)])
            plot_training_history(hist_file, title, level="File")

        elif "LSTM" in title:
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train_f)
            X_val_scaled = scaler.transform(X_val_f)
            X_train_reshaped = X_train_scaled.reshape(X_train_scaled.shape[0], 1, X_train_scaled.shape[1])
            X_val_reshaped = X_val_scaled.reshape(X_val_scaled.shape[0], 1, X_val_scaled.shape[1])
            model_file = Sequential([
                Input(shape=(1, X_train_scaled.shape[1])),
                LSTM(64), Dropout(0.3),
                Dense(32, activation='relu'), Dropout(0.3),
                Dense(3, activation='softmax')
            ])
            model_file.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
            hist_file = model_file.fit(X_train_reshaped, y_train_f, validation_data=(X_val_reshaped, y_val_f),
                                       epochs=50, batch_size=32, verbose=0,
                                       callbacks=[EarlyStopping(patience=5, restore_best_weights=True)])
            plot_training_history(hist_file, title, level="File")

        elif "TabNet" in title:
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train_f)
            X_val_scaled = scaler.transform(X_val_f)
            mlp_file = MLPClassifier(hidden_layer_sizes=(100,50), activation='relu', max_iter=200,
                                     early_stopping=True, validation_fraction=0.2, random_state=RANDOM_STATE)
            mlp_file.fit(X_train_scaled, y_train_f)
            print(f"File-level training for {title} completed (no history plot available for TabNet/MLP).")

        elif "CNN+LSTM" in title:
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train_f)
            X_val_scaled = scaler.transform(X_val_f)
            X_train_reshaped = X_train_scaled.reshape(X_train_scaled.shape[0], X_train_scaled.shape[1], 1)
            X_val_reshaped = X_val_scaled.reshape(X_val_scaled.shape[0], X_val_scaled.shape[1], 1)
            model_file = Sequential([
                Input(shape=(X_train_scaled.shape[1], 1)),
                Conv1D(64, 3, activation='relu', padding='same'), BatchNormalization(),
                Conv1D(32, 3, activation='relu', padding='same'), BatchNormalization(),
                MaxPooling1D(2, padding='same'),   # <-- أضف padding='same'
                LSTM(64, return_sequences=True), Dropout(0.3),
                LSTM(32), Dropout(0.3),
                Dense(64, activation='relu'),
                Dense(3, activation='softmax')
            ])
            model_file.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
            hist_file = model_file.fit(X_train_reshaped, y_train_f, validation_data=(X_val_reshaped, y_val_f),
                                       epochs=50, batch_size=32, verbose=0,
                                       callbacks=[EarlyStopping(patience=5, restore_best_weights=True)])
            plot_training_history(hist_file, title, level="File")

    return model_copy

In [ ]:
def clone_model(model):
    """Return a fresh copy of the model (handles sklearn pipelines and custom deep models)."""

    # If it's a sklearn pipeline (has named_steps), try to clone normally
    if hasattr(model, 'named_steps'):
        try:
            return sklearn_clone(model)
        except Exception:
            pass
    # For LSTMClassifier3
    if hasattr(model, 'named_steps') and isinstance(model.named_steps.get('clf'), LSTMClassifier3):
        # Extract parameters and recreate
        clf = model.named_steps['clf']
        new_clf = LSTMClassifier3(units=clf.units, dropout=clf.dropout,
                                  batch_size=clf.batch_size, epochs=clf.epochs,
                                  optimizer=clf.optimizer)
        return Pipeline([('clf', new_clf)])

    # For TabNetWrapper3
    if hasattr(model, 'named_steps') and isinstance(model.named_steps.get('clf'), TabNetWrapper3):
        clf = model.named_steps['clf']
        new_clf = TabNetWrapper3(n_d=clf.n_d, n_a=clf.n_a, n_steps=clf.n_steps,
                                 gamma=clf.gamma, lr=clf.lr,
                                 max_epochs=clf.max_epochs, patience=clf.patience,
                                 seed=clf.seed)
        return Pipeline([('smote', model.named_steps.get('smote')), ('clf', new_clf)])


    # For MLP (sklearn pipeline), just clone
    try:
        return sklearn_clone(model)
    except:
        return model

In [ ]:
def calculate_metrics(y_true, y_pred, y_proba):
    acc = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
    recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    fnr = 1 - recall
    loss = log_loss(y_true, y_proba, labels=[0,1,2])
    return acc, loss, precision, recall, f1, fnr

def plot_precision_recall_curves(y_true, y_proba, class_names, level="Window"):
    """Plot Precision-Recall curves for each class (one-vs-rest)"""
    n_classes = 3
    plt.figure(figsize=(8,6))
    for i in range(n_classes):
        precision_curve, recall_curve, _ = precision_recall_curve(y_true == i, y_proba[:, i])
        ap_score = average_precision_score(y_true == i, y_proba[:, i])
        plt.plot(recall_curve, precision_curve, lw=2, label=f'{class_names[i]} (AP = {ap_score:.2f})')
    plt.xlabel('Recall')
    plt.ylabel('Precision')
    plt.title(f'Precision-Recall Curves ({level} Level)')
    plt.legend(loc="lower left")
    plt.grid(True)
    plt.show()

def plot_training_history(history, title="Model", level="Window"):
    """Plot training and validation loss & accuracy.
    Supports Keras and TabNet history objects."""
    if history is None:
        print(f"No history to plot for {title}")
        return

    # Extract dictionary
    if hasattr(history, 'history'):
        hist_dict = history.history
    elif isinstance(history, dict):
        hist_dict = history
    else:
        hist_dict = getattr(history, 'history', {})
        if not hist_dict:
            hist_dict = getattr(history, 'history_', {})

    if not hist_dict:
        print(f"History dictionary is empty for {title}")
        return

    print(f"Available history keys for {title}: {list(hist_dict.keys())}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # ----- Loss -----
    train_loss_key = None
    val_loss_key = None
    for key in hist_dict.keys():
        if 'loss' in key.lower() and 'val' not in key.lower():
            train_loss_key = key
            break
    for key in hist_dict.keys():
        if 'val' in key.lower() and 'loss' in key.lower():
            val_loss_key = key
            break

    if train_loss_key:
        axes[0].plot(hist_dict[train_loss_key], label='Train Loss')
    if val_loss_key:
        axes[0].plot(hist_dict[val_loss_key], label='Validation Loss')
    if train_loss_key or val_loss_key:
        axes[0].set_title(f'{title} - Loss ({level} Level)')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].legend()
    else:
        axes[0].text(0.5, 0.5, 'No loss data', ha='center', va='center')

    # ----- Accuracy -----
    train_acc_key = None
    val_acc_key = None
    for key in hist_dict.keys():
        if 'accuracy' in key.lower() and 'val' not in key.lower():
            train_acc_key = key
            break
    for key in hist_dict.keys():
        if 'val' in key.lower() and 'accuracy' in key.lower():
            val_acc_key = key
            break

    if train_acc_key is None and val_acc_key is None:
        axes[1].text(0.5, 0.5, 'No accuracy data', ha='center', va='center')
    else:
        if train_acc_key:
            axes[1].plot(hist_dict[train_acc_key], label='Train Accuracy')
        if val_acc_key:
            axes[1].plot(hist_dict[val_acc_key], label='Validation Accuracy')
        axes[1].set_title(f'{title} - Accuracy ({level} Level)')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Accuracy')
        axes[1].legend()

    plt.tight_layout()
    plt.show()

def get_file_level_features(X_window, y_window, groups_window):
    """
    Aggregate window-level features to file-level by taking mean of features per file.
    Returns:
        X_file: array of shape (n_files, n_features)
        y_file: array of shape (n_files,)
        file_ids: list of unique file IDs
    """
    df = pd.DataFrame(X_window)
    df['y'] = y_window
    df['group'] = groups_window
    # Group by file_id (group) and calculate mean of all features
    X_file = df.groupby('group').mean().drop(['y'], axis=1).values
    y_file = df.groupby('group')['y'].agg(lambda x: x.mode().iloc[0]).values
    file_ids = df.groupby('group').apply(lambda x: x.name).tolist()
    return X_file, y_file, file_ids

In [ ]:
def train_and_evaluate(model_pipeline, param_grid, X, y, groups, title):
    start_time = time.time()
    search = RandomizedSearchCV(model_pipeline, param_grid, n_iter=15,
                                scoring='f1_macro', cv=cv, random_state=RANDOM_STATE,
                                n_jobs=-1, verbose=1)
    search.fit(X, y, groups=groups)
    elapsed = time.time() - start_time
    print(f"Training time for {title}: {elapsed:.2f} seconds")
    best_est = search.best_estimator_
    evaluate_model(best_est, X, y, groups, title=title, training_time=elapsed)
    return best_est

In [ ]:
# LightGBM
lgb_pipe = Pipeline([#("scaler", StandardScaler()),
                     ("smote", SMOTE(random_state=RANDOM_STATE)),
                     ("clf", LGBMClassifier(random_state=RANDOM_STATE, verbose=-1))])
lgb_params = {"clf__n_estimators": [100,200,300], "clf__learning_rate": [0.01,0.05,0.1],
              "clf__num_leaves": [15,31,63], "clf__max_depth": [-1,5,10]}
best_lgb = train_and_evaluate(lgb_pipe, lgb_params, X, y, groups, "LightGBM ")